In [ ]:
from lago.lago import LinkStream, lago_communities
import csv
import pandas as pd

filename = 'data/syn_net.csv'
df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)
time_links = df.values.tolist()

## Initiate empty temporal network (as a link stream)
my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

# NOTE time links can also be imported from txt files with the read_txt() method

## Display linkstream informations
print(f"The link stream consists of {my_linkstream.nb_edges} temporal edges (or time links) accross {my_linkstream.nb_nodes} nodes and {my_linkstream.network_duration} time steps, of which only {my_linkstream.nb_timesteps} contain activity.")


The link stream consists of 193 temporal edges (or time links) accross 20 nodes and 1150 time steps, of which only 193 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_76079/1725866936.py:6: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


In [15]:
df.head(5)

,source,destination,timestamp
0,0,7,9
1,0,2,17
2,0,9,25
3,0,18,28
4,0,14,35


In [17]:
import importlib
from lago.lago import LinkStream, lago_communities

my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

## Compute dynamic communities
dynamic_communities = lago_communities(
    my_linkstream,
    nb_iter=20, # run LAGO 3 times and return best result
    )

# Each dynamic community is represented by a list of (<node>, <time instant>)

print(f"{len(dynamic_communities)} dynamic communities have been found")
print(f"dynamic communities: {dynamic_communities}")
from lago.l_modularity_function import longitudinal_modularity

## Compute Longitudinal Modularity score
## (the higher the better / maximum is 1)
long_mod_score = longitudinal_modularity(
    my_linkstream, 
    dynamic_communities,
    lex_type="MM"
    )

print(f"Longitudinal Modularity score of {long_mod_score} ")

6 dynamic communities have been found
dynamic communities: {0: {(18, 673), (7, 682), (8, 647), (16, 749), (13, 762), (18, 691), (11, 652), (16, 767), (11, 899), (11, 670), (11, 917), (16, 605), (12, 653), (7, 785), (11, 935), (16, 623), (7, 803), (16, 870), (11, 773), (7, 821), (16, 888), (11, 791), (13, 721), (18, 650), (7, 659), (16, 726), (13, 739), (18, 668), (11, 629), (7, 677), (16, 744), (13, 757), (18, 686), (11, 647), (16, 762), (11, 894), (11, 665), (11, 912), (16, 600), (7, 780), (16, 847), (16, 618), (11, 750), (7, 798), (16, 865), (11, 768), (16, 883), (7, 636), (11, 786), (13, 716), (18, 645), (7, 654), (16, 721), (13, 734), (18, 663), (11, 624), (7, 672), (16, 739), (11, 642), (11, 889), (11, 660), (16, 577), (11, 907), (16, 595), (7, 775), (16, 842), (10, 682), (16, 613), (11, 745), (7, 793), (16, 860), (11, 763), (16, 878), (7, 631), (11, 781), (13, 711), (18, 640), (7, 649), (16, 716), (13, 729), (18, 658), (7, 667), (16, 734), (11, 866), (11, 637), (11, 884), (16, 57

In [ ]:
from __future__ import annotations

from typing import Dict, Hashable, Iterable, Tuple, Any, Optional, Union
import pandas as pd

Node = Hashable
Time = Hashable


def commu_dict_to_interaction_csv(
    commu_dict: Dict[Any, Iterable[Tuple[Node, Time]]],
    syn_csv_path: str = "syn_net.csv",
    out_csv_path: str = "labeled_syn_net.csv",
    *,
    source_col: Union[str, int] = "source",
    destination_col: Union[str, int] = "destination",
    timestamp_col: Union[str, int] = "timestamp",
    syn_header: Optional[int] = "infer",
    syn_sep: str = ",",
    syn_skip_first_row: bool = False,

    node_dtype: Any = int,
    time_dtype: Any = int,

    unknown_commu_value: Any = -1,

    on_conflict: str = "error",
) -> pd.DataFrame:

    syn_df = pd.read_csv(
        syn_csv_path,
        sep=syn_sep,
        header=syn_header,
        skiprows=1 if syn_skip_first_row else None,
        usecols=[source_col, destination_col, timestamp_col],
    ).copy()

    syn_df[source_col] = syn_df[source_col].astype(node_dtype)
    syn_df[destination_col] = syn_df[destination_col].astype(node_dtype)
    syn_df[timestamp_col] = syn_df[timestamp_col].astype(time_dtype)

    active_pairs = set(zip(syn_df[source_col].tolist(), syn_df[timestamp_col].tolist()))
    active_pairs |= set(zip(syn_df[destination_col].tolist(), syn_df[timestamp_col].tolist()))

    node_time_to_commu: Dict[Tuple[Node, Time], Any] = {}

    def _assign(k: Tuple[Node, Time], commu: Any):
        if k not in node_time_to_commu:
            node_time_to_commu[k] = commu
            return
        if node_time_to_commu[k] == commu:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            node_time_to_commu[k] = commu
            return
        raise ValueError(f"Conflict: (node,time)={k} appears in multiple communities: "
                         f"{node_time_to_commu[k]} vs {commu}")

    for commu, pairs in commu_dict.items():
        for u, t in pairs:
            u = node_dtype(u)
            t = time_dtype(t)
            k = (u, t)
            if k in active_pairs:
                _assign(k, commu)

    def _lookup(u: int, t: int):
        return node_time_to_commu.get((u, t), unknown_commu_value)

    syn_df["source_commu"] = [
        _lookup(int(s), int(t)) for s, t in zip(syn_df[source_col], syn_df[timestamp_col])
    ]
    syn_df["destination_commu"] = [
        _lookup(int(d), int(t)) for d, t in zip(syn_df[destination_col], syn_df[timestamp_col])
    ]

    out_df = syn_df.rename(
        columns={
            source_col: "source",
            destination_col: "destination",
            timestamp_col: "timestamp",
        }
    )[["source", "destination", "timestamp", "source_commu", "destination_commu"]]

    out_df.to_csv(out_csv_path, index=False)
    return out_df

real_communities = commu_dict_to_interaction_csv(
    dynamic_communities,
    syn_csv_path="data/syn_net.csv",
    out_csv_path="lago_community.csv",
    syn_header="infer", 
    syn_skip_first_row=False,
    unknown_commu_value=-1, 
    on_conflict="error",
)

(real_communities.head(10))

,source,destination,timestamp,source_commu,destination_commu
0,0,7,9,2,2
1,0,2,17,2,2
2,0,9,25,2,2
3,0,18,28,2,2
4,0,14,35,2,2
5,0,6,39,2,2
6,1,2,48,2,2
7,1,3,52,2,2
8,1,13,60,2,2
9,1,14,69,2,2
